In [11]:
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.documents import Document
from langchain_classic.retrievers.multi_query import MultiQueryRetriever
from langchain_huggingface import ChatHuggingFace, HuggingFaceEndpoint

In [6]:
# Relevant health and wellness documents 
allDocs = [
    Document(page_content="Regular exercise can help improve cardiovascular health and boost mood."),
    Document(page_content="A balanced diet rich in fruits and vegetables is essential for overall well-being."),
    Document(page_content="Photosynthesis is the process by which green plants and some other organisms use sunlight to synthesize foods with the help of chlorophyll."),
    Document(page_content="Getting enough sleep is crucial for mental and physical health."),
    Document(page_content="Cricket is a popular sport played worldwide, known for its strategic gameplay and team spirit."),
    Document(page_content="Staying hydrated is important for maintaining energy levels and supporting bodily functions."),
    Document(page_content="Black Holes are regions of spacetime where gravity is so strong that nothing, not even light, can escape from them.")
]

In [7]:
# generating embedding
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5195.65it/s]


In [8]:
# create FAISS vector from documents 
vector_store = FAISS.from_documents(
    allDocs,
    embeddings
)

In [14]:
# create retrieve
similarity_retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={"k":5, "lambda_mut": 1} # top k results , lambda_mut controls the balance between relevance and diversity (0 to 1)
)

In [15]:
# create multi query retriever
multi_query_retriever = MultiQueryRetriever.from_llm(
    retriever = vector_store.as_retriever(search_kwargs={"k":5}),
    llm = ChatHuggingFace(llm=HuggingFaceEndpoint(
        repo_id="meta-llama/Llama-3.1-8B-Instruct",
        task="text-generation",
        max_new_tokens=512,
        temperature=0.1
    ))
)

In [16]:
query = "how to improve energy levels and maintain health?."


In [17]:
similarity_results = similarity_retriever.invoke(query)
multi_query_results = multi_query_retriever.invoke(query)

In [18]:
for i,doc in enumerate(similarity_results):
    print(f"Similarity Result {i+1}: {doc.page_content}")

Similarity Result 1: Regular exercise can help improve cardiovascular health and boost mood.
Similarity Result 2: Staying hydrated is important for maintaining energy levels and supporting bodily functions.
Similarity Result 3: A balanced diet rich in fruits and vegetables is essential for overall well-being.
Similarity Result 4: Getting enough sleep is crucial for mental and physical health.
Similarity Result 5: Photosynthesis is the process by which green plants and some other organisms use sunlight to synthesize foods with the help of chlorophyll.


In [19]:
for i,doc in enumerate(multi_query_results):
    print(f"Multi Query Result {i+1}: {doc.page_content}")

Multi Query Result 1: Black Holes are regions of spacetime where gravity is so strong that nothing, not even light, can escape from them.
Multi Query Result 2: A balanced diet rich in fruits and vegetables is essential for overall well-being.
Multi Query Result 3: Regular exercise can help improve cardiovascular health and boost mood.
Multi Query Result 4: Staying hydrated is important for maintaining energy levels and supporting bodily functions.
Multi Query Result 5: Photosynthesis is the process by which green plants and some other organisms use sunlight to synthesize foods with the help of chlorophyll.
Multi Query Result 6: Getting enough sleep is crucial for mental and physical health.
